In [1]:
# 若尚未安裝需要的套件，可在終端或 notebook 執行：
# !pip install requests beautifulsoup4 lxml

import requests         # 發送 HTTP 請求，用來從網站抓取網頁內容
from bs4 import BeautifulSoup as bs  # 解析 HTML / XML，將字串轉成可以操作的樹狀結構
import lxml             # 上課用的解析器（解析器需已安裝）
import time             # 取得目前時間，或讓程式暫停一段時間，做延遲，避免被 PTT 封鎖
import json             # 處理 JSON 格式的資料（例如 API 回傳或儲存設定）
import re               # 正規表達式

In [18]:
from selenium import webdriver
from selenium.webdriver import ActionChains
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import TimeoutException,ElementNotInteractableException, NoSuchElementException

URL = "https://165dashboard.tw/fraud-method"

#from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup as bs
import pandas as pd
import time

# 設定 Chrome 瀏覽器的選項
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized") # Chrome 瀏覽器在啟動時最大化視窗
options.add_argument("--incognito") # 無痕模式
options.add_argument("--disable-popup-blocking") # 停用 Chrome 的彈窗阻擋功能。

# 建立 Chrome 瀏覽器物件
driver = webdriver.Chrome(options=options)
driver.implicitly_wait(10) # 隱性等待
driver.get(URL)

time.sleep(3)
# 初始化 ActionChains
action = ActionChains(driver)




In [19]:
driver.quit()

In [13]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time


# ==========================================
# 基本設定
# ==========================================

BASE_URL = "https://165dashboard.tw"
URL = f"{BASE_URL}/fraud-method"


# ==========================================
# 啟動 Chrome WebDriver
# ==========================================

options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")

driver = webdriver.Chrome(options=options)

driver.get(URL)

time.sleep(3)


# ==========================================
# 找出所有 p-panel
# ==========================================

all_panels = driver.find_elements(
    By.CSS_SELECTOR,
    "p-panel"
)



# ==========================================
# 只保留真正有標籤名稱的 panel
# ==========================================

panels = []

for panel in all_panels:

    try:

        tag = panel.find_element(
            By.CSS_SELECTOR,
            ".fraud-card__header span"
        )

        tag_name = tag.text.strip()

        if tag_name:
            panels.append(panel)

    except:
        pass


print("標籤數量:", len(panels))


# ==========================================
# 儲存最後結果
# ==========================================

results = []


# ==========================================
# 逐一處理每一個標籤
# ==========================================

for index in range(len(panels)):

    try:

        # ----------------------------------
        # 重新取得所有 panel
        # ----------------------------------

        all_panels = driver.find_elements(
            By.CSS_SELECTOR,
            "p-panel"
        )


        # ----------------------------------
        # 重新找出真正有名稱的 panel
        # ----------------------------------

        valid_panels = []

        for p in all_panels:

            try:

                span = p.find_element(
                    By.CSS_SELECTOR,
                    ".fraud-card__header span"
                )

                name = span.text.strip()

                if name:
                    valid_panels.append(p)

            except:
                pass


        panel = valid_panels[index]


        # ----------------------------------
        # 取得標籤名稱
        # ----------------------------------

        tag = panel.find_element(
            By.CSS_SELECTOR,
            ".fraud-card__header span"
        )

        tag_name = tag.text.strip()


        print("\n")
        print(f"第 {index + 1} 個標籤")
        print(f"標籤名稱: {tag_name}")
        print("==============================")


        # ----------------------------------
        # 找到展開按鈕
        # ----------------------------------

        button = panel.find_element(
            By.CSS_SELECTOR,
            "button.p-panel-toggler"
        )


        # ----------------------------------
        # 檢查目前是否已展開
        # ----------------------------------

        expanded = button.get_attribute(
            "aria-expanded"
        )


        # ----------------------------------
        # 如果尚未展開，就點擊
        # ----------------------------------

        if expanded == "false":

            driver.execute_script(
                "arguments[0].click();",
                button
            )

            time.sleep(0.8)


        # ----------------------------------
        # 取得 content ID
        # ----------------------------------

        content_id = button.get_attribute(
            "aria-controls"
        )

        print("Content ID:", content_id)


        # ----------------------------------
        # Selenium 找到對應的 content
        # ----------------------------------

        content = driver.find_element(
            By.ID,
            content_id
        )


        # ----------------------------------
        # 使用 BeautifulSoup 解析 HTML
        # ----------------------------------

        html = content.get_attribute(
            "outerHTML"
        )

        soup = BeautifulSoup(
            html,
            "html.parser"
        )


        # ==================================
        # 找「前往防詐秘笈」
        # ==================================

        link = None

        for a in soup.find_all("a"):

            a_text = a.get_text(
                strip=True
            )

            if "前往防詐秘笈" in a_text:

                link = a
                break


        # ==================================
        # 取得 URL
        # ==================================

        full_url = None

        if link:

            href = link.get("href")

            if href:

                full_url = urljoin(
                    BASE_URL,
                    href
                )


                print("✅ 找到:")

                print(
                    (
                        tag_name,
                        full_url
                    )
                )

        else:

            print(
                "❌ 找不到「前往防詐秘笈」"
            )


        # ==================================
        # 取得犯罪手法文字
        # ==================================

        body = soup.find(
            "div",
            class_="fraud-card__body"
        )


        content_text = ""


        if body:

            # ------------------------------
            # 移除「前往防詐秘笈」
            # ------------------------------

            link_div = body.find(
                "div",
                class_="fraud-card__link"
            )

            if link_div:

                link_div.decompose()


            # ------------------------------
            # 找「常見犯罪手法及話術如下」
            # ------------------------------

            hint = body.find(
                "div",
                class_="fraud-card__hint"
            )


            if hint:

                # 取得標題
                hint_text = hint.get_text(
                    strip=True
                )

                # 印出標題
                print("\n" + hint_text)


                # 移除標題
                hint.decompose()


            # ------------------------------
            # 取得剩下的文字
            # ------------------------------

            content_text = body.get_text(
                "\n",
                strip=True
            )


            # ------------------------------
            # 印出犯罪手法
            # ------------------------------

            print(content_text)


        else:

            print(
                "❌ 找不到 fraud-card__body"
            )


        # ==================================
        # 儲存結果
        # ==================================

        result = (
            tag_name,
            full_url,
        )


        results.append(result)


    except Exception as e:

        print(
            f"\n❌ 第 {index + 1} 個標籤發生錯誤:"
        )

        print(e)


# ==========================================
# List → Tuple
# ==========================================

results = tuple(results)


# ==========================================
# 輸出完整 Tuple
# ==========================================

print("\n")
print("==========================================")
print("所有結果")
print("==========================================")


for result in results:

    print(result)




# ==========================================
# 關閉瀏覽器
# ==========================================

driver.quit()

標籤數量: 23


第 1 個標籤
標籤名稱: 網路購物詐騙
Content ID: p-panel-0-content
✅ 找到:
('網路購物詐騙', 'https://165dashboard.tw/fraud-method/329818012855046144')

常見犯罪手法及話術如下：
1、珠寶買賣直播
2、賭石直播
3、緬甸戰爭大量便宜珠寶出售
4、優惠價購買餐券
5、便宜價二手車
6、LV包包特賣會
7、代購演唱會門票
8、要求以LINE私下交易
9、先匯款保留商品
10、提供一頁式網站下單
11、商品可面交，但都在偏鄉，或面交時間約不到，只好改成匯款後寄送
12、提供多個不同帳戶供匯款
13、匯款後即消失或封鎖
14、寄出之商品與廣告不符且明顯低價
15、先要求超商繳費，再以金流有問題等理由，要求匯款才能出貨及返還金額


第 2 個標籤
標籤名稱: 假交友(投資詐財)詐騙
Content ID: p-panel-2-content
✅ 找到:
('假交友(投資詐財)詐騙', 'https://165dashboard.tw/fraud-method/312064826387992576')

常見犯罪手法及話術如下：
1、佯稱異國單身異性，並稱自己從事海外高薪工作，並投資高報酬事業
2、社群媒體上照片皆名牌服飾、名錶、名車及香檳名酒
3、聲稱企業主管，常至高檔餐廳用餐，並傳送美食照
4、共同投資，追求財富自由
5、合租高級AI算力機
6、租用泰達幣虛擬礦機可穩定獲利
7、虛擬貨幣平臺技術員，知悉大戶投資虛擬貨幣進場時間並且一同獲利
8、經營買賣紅酒、茶葉等生意，賺取差價，能額外加薪
9、教導投注大陸香港六合彩或大陸彩票等標的
10、佯稱投資專家，能代為操盤投資國外的石油、天然氣買賣
11、假稱要共同經營網路商店，邀約儲值、投資進行獲利
12、稱能賺取販售商品之價差
13、提領獲利需要高級認證
14、繳納處理風險工本費
15、稱境外資金查核嚴格，恐遭懷疑洗錢，要求投入保證金
16、要求繳納出貨費用、保證金，稱若不繳納則先前的獲利皆無法提領
17、邀約一同投資某虛擬貨幣，稱能獲得高額回報，並給予未來美好想像
18、稱自己從事藝術品買賣，邀約一起投資NFT等虛擬商品
19、邀約一同經營網購商店
20、稱自己公司開發機器人，邀約入股投資


第 3 

In [ ]:
from selenium import webdriver
from bs4 import BeautifulSoup
import time


# ============================================================
# 第一支爬蟲取得的 URL
# ============================================================

fraud_urls = (
    ('網路購物詐騙', 'https://165dashboard.tw/fraud-method/329818012855046144'),
    ('假投資詐騙', 'https://165dashboard.tw/fraud-method/327303345054158848'),
    ('釣魚連結詐騙', 'https://165dashboard.tw/fraud-method/320803405264064512'),
    ('假中獎通知詐騙', 'https://165dashboard.tw/fraud-method/322206935585656832'),
    ('假求職詐騙', 'https://165dashboard.tw/fraud-method/325471681525059584'),
    ('假借銀行貸款詐騙', 'https://165dashboard.tw/fraud-method/270466355268947968'),
    ('騙取金融帳戶(卡片)詐騙', 'https://165dashboard.tw/fraud-method/320399472645378048'),
    ('猜猜我是誰詐騙', 'https://165dashboard.tw/fraud-method/307801223623348224'),
    ('假慈善機關(急難救助)詐騙', 'https://165dashboard.tw/fraud-method/289656464379744256'),
    ('收購郵銀簿(電信門號)詐騙', 'https://165dashboard.tw/fraud-method/338873881450254336'),
    ('六合彩詐騙', 'https://165dashboard.tw/fraud-method/286781286448631808'),
    ('信用卡遭盜刷詐騙', 'https://165dashboard.tw/fraud-method/336010695290130432'),
    ('假交友(投資詐財)詐騙', 'https://165dashboard.tw/fraud-method/312064826387992576'),
    ('色情應召詐財詐騙', 'https://165dashboard.tw/fraud-method/321136076590682112'),
    ('假檢警詐騙', 'https://165dashboard.tw/fraud-method/325103882076164096'),
    ('假買家騙賣家詐騙', 'https://165dashboard.tw/fraud-method/334882749754118144'),
    ('假交友(徵婚詐財)詐騙', 'https://165dashboard.tw/fraud-method/332367662917095424'),
    ('假廣告詐騙', 'https://165dashboard.tw/fraud-method/328005776549351424'),
    ('盜(冒)用 LINE 帳號詐騙', 'https://165dashboard.tw/fraud-method/205961288457654272'),
    ('虛擬遊戲詐騙', 'https://165dashboard.tw/fraud-method/330543633121939456'),
    ('假預付型消費詐騙', 'https://165dashboard.tw/fraud-method/343281639280873472'),
    ('假推銷詐騙', 'https://165dashboard.tw/fraud-method/223628654511394816'),
    ('假快遞詐騙', 'https://165dashboard.tw/fraud-method/184119756654120960')
)


# ============================================================
# Selenium
# ============================================================

options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")

driver = webdriver.Chrome(
    options=options
)


# ============================================================
# 儲存全部結果
# ============================================================

all_results = []


# ============================================================
# 開始爬取
# ============================================================

for index, (tag_name, url) in enumerate(
    fraud_urls,
    start=1
):

    print("\n")
    print("=" * 60)
    print(f"第 {index} 個標籤")
    print(f"標籤名稱: {tag_name}")
    print(f"網址: {url}")
    print("=" * 60)


    # ========================================================
    # 開啟網頁
    # ========================================================

    driver.get(url)

    time.sleep(4)


    # ========================================================
    # 取得 HTML
    # ========================================================

    html = driver.page_source


    # ========================================================
    # BeautifulSoup
    # ========================================================

    soup = BeautifulSoup(
        html,
        "html.parser"
    )


    # ========================================================
    # 1. 防詐小撇步 / 詐騙話術解析
    # ========================================================

    detail_results = []

    detail_cards = soup.find_all(
        "div",
        class_="detail-card"
    )


    for card in detail_cards:

        title = card.find(
            "div",
            class_="detail-card__title"
        )

        content = card.find(
            "div",
            class_="detail-card__content"
        )


        if title and content:

            title_text = title.get_text(
                " ",
                strip=True
            )

            content_text = content.get_text(
                "\n",
                strip=True
            )


            print("\n")
            print(f"【{title_text}】")
            print("-" * 42)
            print(content_text)


            detail_results.append(
                (
                    title_text,
                    content_text
                )
            )


    # ========================================================
    # 2. 抓取案例
    # ========================================================

    case_title = ""
    case_content = ""


    # --------------------------------------------------------
    # 找所有案例區塊
    # --------------------------------------------------------

    case_details = soup.find_all(
        "div",
        class_="case-detail__content"
    )


    # --------------------------------------------------------
    # 逐一檢查案例區塊
    # --------------------------------------------------------

    for case_detail in case_details:

        # ----------------------------------------------------
        # 找標題區塊
        # ----------------------------------------------------

        title_block = case_detail.find(
            "div",
            class_="title-block"
        )


        # ----------------------------------------------------
        # 找內容區塊
        # ----------------------------------------------------

        content_block = case_detail.find(
            "div",
            class_="content-block"
        )


        # 沒有內容就跳過
        if not content_block:
            continue


        # ----------------------------------------------------
        # 取得案例內容
        # ----------------------------------------------------

        current_content = content_block.get_text(
            "\n",
            strip=True
        )


        # 內容是空的就跳過
        if not current_content:
            continue


        # ----------------------------------------------------
        # 取得案例標題
        # ----------------------------------------------------

        if title_block:

            title = title_block.find(
                "div",
                class_="title"
            )

        else:

            title = None


        # 沒有標題就跳過
        if not title:
            continue


        current_title = title.get_text(
            " ",
            strip=True
        )


        # ----------------------------------------------------
        # 找到案例
        # ----------------------------------------------------

        case_title = current_title
        case_content = current_content

        break


    # ========================================================
    # 3. 印出案例
    # ========================================================

    print("\n")
    print("【案例】")
    print("------------------------------------------")


    if case_title:

        print(
            f"標題: {case_title}"
        )

    else:

        print(
            "找不到案例標題"
        )


    print()


    if case_content:

        print(
            "內容:"
        )

        print(
            case_content
        )

    else:

        print(
            "找不到案例內容"
        )


    # ========================================================
    # 4. 儲存資料
    # ========================================================

    result = {

        "tag_name": tag_name,

        "url": url,

        "details": detail_results,

        "case_title": case_title,

        "case_content": case_content

    }


    all_results.append(
        result
    )


# ============================================================
# 關閉 Selenium
# ============================================================

driver.quit()


# ============================================================
# 完整結果
# ============================================================

# print("\n")
# print("=" * 60)
# print("完整結果")
# print("=" * 60)


# for result in all_results:

#     print(result)



第 1 個標籤
標籤名稱: 網路購物詐騙
網址: https://165dashboard.tw/fraud-method/329818012855046144


【防詐小撇步】
------------------------------------------
1、	賣貨便、物流平台不會要求加入私人 LINE 客服處理交易。
2、	注意網址與官方網站是否一致，若發現異狀，應立即停止操作。
3、	網路購物不需要透過網銀轉帳來完成「認證」或「帳戶重啟」。
4、	無卡提款碼、網銀操作都是金流授權行為，切勿依陌生人指示操作。
5、	遇到購物糾紛或系統問題，應直接透過官方客服查證。


【詐騙話術解析】
------------------------------------------
1、	詐騙集團在社群平台刊登熱門商品吸引買家上鉤。
2、	架設仿冒網站，讓被害人誤以為進入官方交易系統。
3、	佯稱系統異常、實名認證未完成等理由，誘導加入假客服 LINE，由專人一步步操控被害人。
4、	謊稱進行帳戶驗證，實際上是在引導轉帳及提款。


【案例】
------------------------------------------
標題: 誤入詐騙戰鬥盤　積蓄慘遭爆裂衝擊

內容:
某天在家裡滑手機，知名社群平台上看到一個帳號貼文販售新款戰鬥陀螺。我曾眼睜睜看著這款陀螺在電商平台被秒殺，自己也無暇到實體店面徹夜排隊購買，如今眼前又出現入手機會，不想再扼腕，馬上留言表明購買意願。
沒多久，對方私訊回復傳來一組網址，聲稱是7-11賣貨便交易連結。由於我過去也曾有賣貨便購物經驗，不疑有他，直接點進連結填寫收件與聯絡資料。
資料填完後，頁面突然跳出訊息，顯示我尚未完成「實名認證」，無法完成交易。接著畫面又出現一個 LINE 客服連結，要我加入客服處理認證問題。
我加入客服人員後，對方立即回訊表示，只要配合進行帳戶重啟程序，就不需要重新驗證身分，可以直接完成交易。客服先要求我打開網路銀行APP，然後指示我各種花式操作，強調驗證程序不會真的扣款。
當時我雖有些疑惑，一心只想趕緊入手新款陀螺，就這樣轉出數萬元款項而不自知。對方隨後又藉口要求提供無卡提款的「序號」與「預約密碼」作為系統認證。
等我從撿到寶的激情狀態冷靜下來，這才越想越不對勁。重新查看對方提供的網站與客服資料，赫然

In [19]:
# ========================================================
# 抓取案例
# ========================================================

case_title = ""
case_content = ""


# --------------------------------------------------------
# 找出所有案例區塊
# --------------------------------------------------------

case_details = soup.find_all(
    "div",
    class_="case-detail__content"
)


print(
    f"\n找到 {len(case_details)} 個案例區塊"
)


# ========================================================
# 逐一檢查
# ========================================================

for i, case_detail in enumerate(
    case_details,
    start=1
):

    print("\n")
    print(f"========== 案例區塊 {i} ==========")


    # ----------------------------------------------------
    # 找 title-block
    # ----------------------------------------------------

    title_block = case_detail.find(
        "div",
        class_="title-block"
    )


    # ----------------------------------------------------
    # 找 content-block
    # ----------------------------------------------------

    content_block = case_detail.find(
        "div",
        class_="content-block"
    )


    # ----------------------------------------------------
    # DEBUG
    # ----------------------------------------------------

    print(
        "title-block:",
        "找到" if title_block else "沒有"
    )

    print(
        "content-block:",
        "找到" if content_block else "沒有"
    )


    # ----------------------------------------------------
    # 如果沒有 content-block
    # 就不是我們要的案例
    # ----------------------------------------------------

    if not content_block:

        print("❌ 不是完整案例，跳過")

        continue


    # ----------------------------------------------------
    # 找案例標題
    # ----------------------------------------------------

    if title_block:

        title = title_block.find(
            "div",
            class_="title"
        )

    else:

        title = None


    if not title:

        print("❌ 找不到案例標題")

        continue


    current_title = title.get_text(
        " ",
        strip=True
    )


    # ----------------------------------------------------
    # 找案例內容
    # ----------------------------------------------------

    current_content = content_block.get_text(
        "\n",
        strip=True
    )


    # ----------------------------------------------------
    # 確認內容不是空的
    # ----------------------------------------------------

    if not current_content:

        print("❌ content-block 是空的")

        continue


    # ====================================================
    # 找到真正案例
    # ====================================================

    case_title = current_title

    case_content = current_content


    print("\n")
    print("✅ 找到真正案例")
    print("------------------------------------------")
    print(
        f"標題: {case_title}"
    )

    print("\n內容:")
    print(
        case_content
    )


    # 找到後就停止
    break


# ========================================================
# 如果完全沒有找到
# ========================================================

if not case_title or not case_content:

    print("\n")
    print("❌ 找不到真正的案例內容")


找到 3 個案例區塊


========== 案例區塊 1 ==========
title-block: 找到
content-block: 找到
❌ content-block 是空的


========== 案例區塊 2 ==========
title-block: 找到
content-block: 找到


✅ 找到真正案例
------------------------------------------
標題: 假包裹真詐騙 差點成為冤大頭

內容:
我平時習慣網路購物，某天手機傳來一封陌生號碼的簡訊，通知我商品送抵超商門市，要求盡速領貨。記憶中，我的確有下訂一組保養品，所以，沒有多想，下班後就到住家附近的超商領貨，怎知道居然有兩件包裹掛在我名下，一件寄貨者是保養品商家名稱，另一件寄貨者名叫「天天向上」、內容物標示「生活用品」，可是我壓根沒印象有訂什麼生活用品。
超商櫃台等待結帳的隊伍因為我的猶豫而越排越長，為避免耽誤大家的時間，我趕緊將兩件包裹結帳離開，於返家途中，百思不得其解「這莫名包裹到底何時下訂的？」到家後，我立刻將那奇怪的包裹開箱，結果是一罐寫著簡體中文的廉價護髮油，連忙用手機掃描包裹外的賣家QR Code，竟然顯示亂碼，根本沒有賣家資訊！
我帶著假包裹返回超商詢問店員能否退貨？店員很同情我的遭遇，協助我透過快遞系統申請爭議包裹的退貨與退費，恰好店內有警察進來巡邏，上前關切狀況，提醒我這是典型的假包裹詐騙手法，詐騙集團透過快遞業者或超商店到店服務，以貨到付款方式寄送廉價商品給不特定民眾，民眾不察上當取貨付款，多數人因為被騙金額不大，也不願多做追究。
幸好，透過店員與警方的協助，幾天後順利收到快遞業者的退費，這次教訓也提醒我，下次收取包裹付款前一定要確認賣家資訊真偽，以免又傻傻地被假包裹給騙錢了。


In [ ]:
fraud_urls = (
('網路購物詐騙', 'https://165dashboard.tw/fraud-method/329818012855046144'),
('假投資詐騙', 'https://165dashboard.tw/fraud-method/327303345054158848'),
('釣魚連結詐騙', 'https://165dashboard.tw/fraud-method/320803405264064512'),
('假中獎通知詐騙', 'https://165dashboard.tw/fraud-method/322206935585656832'),
('假求職詐騙', 'https://165dashboard.tw/fraud-method/325471681525059584'),
('假借銀行貸款詐騙', 'https://165dashboard.tw/fraud-method/270466355268947968'),
('騙取金融帳戶(卡片)詐騙', 'https://165dashboard.tw/fraud-method/320399472645378048'),
('猜猜我是誰詐騙', 'https://165dashboard.tw/fraud-method/307801223623348224'),
('假慈善機關(急難救助)詐騙', 'https://165dashboard.tw/fraud-method/289656464379744256'),
('收購郵銀簿(電信門號)詐騙', 'https://165dashboard.tw/fraud-method/338873881450254336'),
('六合彩詐騙', 'https://165dashboard.tw/fraud-method/286781286448631808'),
('信用卡遭盜刷詐騙', 'https://165dashboard.tw/fraud-method/336010695290130432'),
('假交友(投資詐財)詐騙', 'https://165dashboard.tw/fraud-method/312064826387992576'),
('色情應召詐財詐騙', 'https://165dashboard.tw/fraud-method/321136076590682112'),
('假檢警詐騙', 'https://165dashboard.tw/fraud-method/325103882076164096'),
('假買家騙賣家詐騙', 'https://165dashboard.tw/fraud-method/334882749754118144'),
('假交友(徵婚詐財)詐騙', 'https://165dashboard.tw/fraud-method/332367662917095424'),
('假廣告詐騙', 'https://165dashboard.tw/fraud-method/328005776549351424'),
('盜(冒)用 LINE 帳號詐騙', 'https://165dashboard.tw/fraud-method/205961288457654272'),
('虛擬遊戲詐騙', 'https://165dashboard.tw/fraud-method/330543633121939456'),
('假預付型消費詐騙', 'https://165dashboard.tw/fraud-method/343281639280873472'),
('假推銷詐騙', 'https://165dashboard.tw/fraud-method/223628654511394816'),
('假快遞詐騙', 'https://165dashboard.tw/fraud-method/184119756654120960')
)